In [5]:
import sys
sys.path.append('..')
from src import *
import pandas as pd
import sqlite3


In [ ]:
conn = sqlite3.connect('../data/finance.db')
init_db(conn)

In [49]:
from datetime import datetime, timedelta

prices = pd.read_sql("SELECT * FROM price_history WHERE interval = '1d'", conn)
prices['datetime'] = pd.to_datetime(prices['datetime'])

cutoff = datetime.now() - timedelta(days=365)
prices = prices[prices['datetime'] >= cutoff].sort_values(['ticker', 'datetime'])

avg_volume = prices.groupby('ticker')['Volume'].mean().reset_index(name='avg_volume')

In [42]:
sectors = pd.read_sql("SELECT ticker, sector FROM tickers", conn)
avg_volume = avg_volume.merge(sectors, on='ticker').dropna(subset=['sector'])

top5_by_volume = (avg_volume
                   .sort_values('avg_volume', ascending=False)
                   .groupby('sector')
                   .head(10)
                   .sort_values(['sector', 'avg_volume'], ascending=[True, False]))
top5_by_volume

,ticker,avg_volume,sector
427,T,4.868622e+07,Communication Services
330,NFLX,4.287096e+07,Communication Services
95,CMCSA,3.374147e+07,Communication Services
211,GOOGL,3.211883e+07,Communication Services
480,WBD,2.999740e+07,Communication Services
...,...,...,...
411,SO,5.781388e+06,Utilities
125,D,5.638815e+06,Utilities
101,CNP,5.503720e+06,Utilities
495,XEL,5.169838e+06,Utilities


In [50]:
# top_tickers = top5_by_volume['ticker'].tolist()
# prices = prices[prices['ticker'].isin(top_tickers)]
prices['daily_return'] = prices.groupby('ticker')['Close'].pct_change()

print(prices['ticker'].nunique(), "of", len(top_tickers), "selected tickers have price data")

503 of 110 selected tickers have price data


In [51]:
summary = prices.groupby('ticker')['daily_return'].agg(
    avg_daily_return='mean',
    volatility='std',
).reset_index()

summary['annualized_return'] = summary['avg_daily_return'] * 252
summary['annualized_volatility'] = summary['volatility'] * (252 ** 0.5)
summary = summary.merge(top5_by_volume, on='ticker')
summary = summary.sort_values(['sector', 'avg_volume'], ascending=[True, False])
summary

,ticker,avg_daily_return,volatility,annualized_return,annualized_volatility,avg_volume,sector
94,T,-0.000094,0.016053,-0.023578,0.254834,4.868622e+07,Communication Services
73,NFLX,-0.001852,0.023066,-0.466592,0.366163,4.287096e+07,Communication Services
16,CMCSA,-0.000793,0.020393,-0.199847,0.323729,3.374147e+07,Communication Services
42,GOOGL,0.001566,0.020074,0.394619,0.318657,3.211883e+07,Communication Services
104,WBD,0.002033,0.017206,0.512235,0.273134,2.999740e+07,Communication Services
...,...,...,...,...,...,...,...
92,SO,-0.000377,0.010919,-0.095087,0.173326,5.781388e+06,Utilities
29,D,0.000208,0.012793,0.052431,0.203083,5.638815e+06,Utilities
18,CNP,-0.000018,0.010824,-0.004531,0.171826,5.503720e+06,Utilities
108,XEL,-0.000396,0.011847,-0.099791,0.188071,5.169838e+06,Utilities


In [52]:
def max_drawdown(close_prices):
    cumulative_max = close_prices.cummax()
    drawdown = (close_prices - cumulative_max) / cumulative_max
    return drawdown.min()

drawdowns = prices.groupby('ticker')['Close'].apply(max_drawdown).reset_index(name='max_drawdown')
summary = summary.merge(drawdowns, on='ticker')
summary

,ticker,avg_daily_return,volatility,annualized_return,annualized_volatility,avg_volume,sector,max_drawdown
0,T,-0.000094,0.016053,-0.023578,0.254834,4.868622e+07,Communication Services,-0.288854
1,NFLX,-0.001852,0.023066,-0.466592,0.366163,4.287096e+07,Communication Services,-0.455432
2,CMCSA,-0.000793,0.020393,-0.199847,0.323729,3.374147e+07,Communication Services,-0.312078
3,GOOGL,0.001566,0.020074,0.394619,0.318657,3.211883e+07,Communication Services,-0.210472
4,WBD,0.002033,0.017206,0.512235,0.273134,2.999740e+07,Communication Services,-0.156771
...,...,...,...,...,...,...,...,...
105,SO,-0.000377,0.010919,-0.095087,0.173326,5.781388e+06,Utilities,-0.155308
106,D,0.000208,0.012793,0.052431,0.203083,5.638815e+06,Utilities,-0.150663
107,CNP,-0.000018,0.010824,-0.004531,0.171826,5.503720e+06,Utilities,-0.182086
108,XEL,-0.000396,0.011847,-0.099791,0.188071,5.169838e+06,Utilities,-0.153881


In [53]:
pivot = prices.pivot(index='datetime', columns='ticker', values='daily_return')
correlation = pivot.corr()
correlation

ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
ticker,,,,,,,,,,,,,,,,,,,,,
A,1.000000,0.088473,0.151984,0.283695,0.207355,-0.020236,0.179982,0.140767,0.222782,0.007922,...,0.194309,0.198428,-0.023103,-0.146275,0.213116,0.294870,0.024973,0.233961,0.202699,0.206796
AAPL,0.088473,1.000000,0.107773,0.267882,0.110784,0.109922,0.145495,0.228914,0.162929,-0.003306,...,-0.005581,0.214541,0.002302,-0.101839,0.128392,0.221174,0.115439,0.088381,0.205349,0.119777
ABBV,0.151984,0.107773,1.000000,0.029138,0.284319,0.264367,0.154717,0.111619,-0.105243,0.054185,...,0.081905,-0.102915,0.235941,0.021043,0.100282,0.011471,0.170593,0.153882,0.008780,0.072255
ABNB,0.283695,0.267882,0.029138,1.000000,0.179225,0.102344,0.341536,0.383960,0.166866,-0.082300,...,0.092455,0.362532,-0.030090,-0.259050,0.185605,0.423072,0.078276,0.211426,0.246292,0.205406
ABT,0.207355,0.110784,0.284319,0.179225,1.000000,0.289458,0.264629,0.199867,-0.138653,0.026172,...,0.224614,0.125377,0.113796,0.014189,0.181897,0.015440,0.257559,0.350301,0.045246,0.331573
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XYZ,0.294870,0.221174,0.011471,0.423072,0.015440,-0.048381,0.317267,0.332391,0.207694,-0.118229,...,0.080016,0.232094,-0.120787,-0.221978,0.178168,1.000000,-0.004849,0.104830,0.244290,0.183449
YUM,0.024973,0.115439,0.170593,0.078276,0.257559,0.312381,0.115506,0.057984,-0.030272,-0.025786,...,0.197510,0.171592,0.177484,-0.020794,0.111435,-0.004849,1.000000,0.192209,0.115077,0.031712
ZBH,0.233961,0.088381,0.153882,0.211426,0.350301,0.285429,0.166344,0.171883,0.011361,0.103501,...,0.259474,0.063802,0.165477,0.017150,0.261334,0.104830,0.192209,1.000000,0.134585,0.291745


In [55]:
correlation['INTC'].sort_values(ascending=True)

ticker
BRO    -0.407900
RSG    -0.386883
AJG    -0.363996
CB     -0.359120
MRSH   -0.351853
          ...   
AMAT    0.587212
KLAC    0.600425
LRCX    0.610944
AMD     0.617251
INTC    1.000000
Name: INTC, Length: 503, dtype: float64

In [28]:
!{sys.executable} -m pytest -x -v

============================= test session starts ==============================
platform darwin -- Python 3.11.13, pytest-9.1.1, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/finance-pipeline/bin/python
cachedir: .pytest_cache
rootdir: /Users/Kokiri/Documents/github/finance-data-pipeline
configfile: pyproject.toml
collected 0 items                                                              

============================ no tests ran in 0.00s =============================


In [25]:
from finance_data_pipeline import analysis as analysis_module
from finance_data_pipeline.analysis import (
    get_price_technicals,
    get_fundamentals,
    get_analyst_view,
    get_options_summary,
    analyze_stock,
    print_report,
    plot_technicals,
)